# Change json element - loops files

In [1]:
import ollama
embed_model='nomic-embed-text', # 
embed_model = 'qwen3-embedding:0.6b'

response = ollama.embed(
    model=embed_model,
    input='The quick brown fox jumps over the lazy dog.'
)

# Access the numerical vector array
embedding = response['embeddings'][0]
print(embedding[:3])


[0.024083033, -0.048990417, -0.0032610635]


In [13]:
import json
import os
from pathlib import Path
import ollama
from ollama import chat
import numpy as np
import time

class utils():
    def get_base_dir():
        return os.getcwd()
        
    def check_dir_exist(dir):
        if not os.path.exists(dir):
            os.makedirs(dir)
        return True
    
    def check_file_exists(file_path):
        if os.path.exists(file_path):
            return True
        else:
            return False
    
    def clean_garbage(input_text):
        print(len(input_text) )
        garbage = "\n\nSome case metadata and case summaries were written with the help of AI, which can produce inaccuracies. You\n        should read the full case before relying on it for legal research purposes.\n        This site is protected by reCAPTCHA and the Google\n    Privacy Policy and\n    Terms of Service apply.\n\n\n\nYou're all set! You already receive all suggested Justia Opinion Summary Newsletters. You can explore additional available newsletters here.\n        \n\n\n                Sign up for our free summaries and get the latest delivered directly to you.\n            \n\n                Our Suggestions:\n            \n\n\n\n\n\n\nEnter Your Email\n\n\n\nEnter Your Email\nEnter Your Email\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\n\n\n\n\n\n\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\nGoogle Scholar\nGoogle Books\nLegal Blogs  \n\n\nGoogle Web \nBing Web \n\n\nGoogle News \nGoogle News Archive \nYahoo! News\nHave a legal question? Get free answers from experienced lawyers!\n\n                Ask Question\nLawyers - Get Listed Now!\nGet a free directory profile listing\n\n\nLawyers - Get Listed Now!\nGet a free directory profile listing"
        output = input_text.replace(garbage, "")
        print(len(output) )
        
        output = output.replace("\n", "").replace("-","").replace("\t", "").replace("(","").replace(")","").replace("[","").replace("]","")
        print(len(output) )
        return output


class AI():
    def get_embeddings(text_input, dimension):
        embed_model = 'qwen3-embedding:0.6b'
        response = ollama.embed(
            model=embed_model,
            input= text_input,
            options={
                "embedding_options": {
                    "dimensions": dimension
                }
            }
        )
        full_embedding = []
        
        # Access the numerical vector array
        try: 
            #print(text_input, "\nresponse['embeddings'][0] == ", response['embeddings'][0], "\n=====================================================")
            full_embedding = response['embeddings'][0]
        except:
            #print(text_input, "\nresponse['embeddings'] == ", response['embeddings'], "\n=====================================================")
            full_embedding = response['embeddings']
        
        truncated_vector = np.array(full_embedding[:dimension])
        
        # 4. Normalize the truncated vector for cosine similarity search
        normalized_vector = truncated_vector / np.linalg.norm(truncated_vector)
        
        return normalized_vector.tolist()

    def get_summary(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Write a 4 sentence summary of this case. "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        # full_response = output.message.content
        full_response = output['response']
        output = []
        return full_response

    def get_judge_and_ruling(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Who were the attorneys and who was the judge, and what was the ruling of the case? "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        full_response = output['response']
        output = []
        return full_response


In [30]:
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
# 9/23/2026 --> multithreading

"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

data3 ->    1. embeddings for 3 fields [court_name, docket_number, state]
            2. summary of big text field
"""

i = 0

def ETL_files(read_file, write_file):

    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        # 9/19/26 - change element name
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")

        # 9/19/26 - get rid of date unused field
        if "date" in json_data:
            json_data.pop("date")

        # 9/19/26 - add state field
        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()

        # 9/21/26 - use AI to pull out data and create more fields
        combo_field = json_data["case_name"] + " " + json_data["case_date"] + " " + json_data["court_name"] 
        json_data["judge_ruling"] = AI.get_judge_and_ruling(combo_field)
        json_data["summary"] = AI.get_summary(combo_field)
    
        json_data["dv_general_text"] = AI.get_embeddings(combo_field + json_data["summary"], 50)
        json_data["dv_judge_ruling"] = AI.get_embeddings(json_data["judge_ruling"], 50)
        
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
 
    """i+=1
    if i > 4:
        break"""


def loop_file_list(file_list):
    writedir = "/Users/anria/ab_python/django/data3/"

    # Loop through all text files
    for read_file in file_list:
        write_file = writedir + read_file.name
        if utils.check_file_exists(write_file):
            continue 
        ETL_files(read_file, write_file)

directory = Path("/Users/anria/ab_python/django/data/")

file_list = directory.glob("*.json")
# print("TYPE ", file_list, list(file_list) )
# Using ThreadPoolExecutor to run tasks concurrently
#with ThreadPoolExecutor(max_workers=1) as executor:
#    results = list(executor.map(loop_file_list, file_list ))
loop_file_list(file_list)

print("Out of loop")

Out of loop


In [ ]:
   ''' i+=1
    if i > 3:
        break '''

In [23]:
print( AI.get_embeddings(" WL 264718 (OH App) *10*")[:3] )

[0.014575815740131797, -0.036081385228413, -0.013378690088046555]


# Kaput -- Done
raise SystemExit  # Stops the cell and prevents further cells from running

In [21]:
raise SystemExit  # Stops the cell and prevents further cells from running


SystemExit: 

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [14]:
raise SystemExit  # Stops the cell and prevents further cells from running
import json
from pathlib import Path
# 9/19/2026. 
"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

"""


directory = Path("/Users/anria/ab_python/fix search/justia/data")
writedir = "/Users/anria/ab_python/fix search/justia/data2/"
i = 0
# Loop through all text files
for read_file in directory.glob("*.json"):
    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")
        if "date" in json_data:
            json_data.pop("date")

        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()
        
        write_file = writedir + read_file.name
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
     
    ''' i+=1
    if i > 3:
        break '''
print("Out of loop")
    

SystemExit: 

In [13]:
"""
9/22/2026
1. Copy the files that landed in django/data3 from original fix search/data to django/data
2. so we have a very reliable 1,500 doc dataset
"""
raise SystemExit  # Stops the cell and prevents further cells from running

import json
from pathlib import Path

fix_search_data_read_dir = '/Users/anria/ab_python/fix search/justia/data/'
django_data3_read_dir = Path('/Users/anria/ab_python/fix search/justia/data3')
django_data_write_dir =      '/Users/anria/ab_python/django/data/'

# json_files = list(fix_search_data3_read_dir.glob("*.json"))

for file in django_data3_read_dir.glob("*.json"):
    read_file = Path(fix_search_data_read_dir + file.name)
    write_file = django_data_write_dir + read_file.name
    """
    print("filename", file.name)
    print("copy_file.name", read_file.name)
    print("copy_file.name", read_file)
    print("write_file.name", write_file)
    break """

    with open(read_file) as file:
        json_data = json.load(file)

        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)

SystemExit: 

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
raise SystemExit  # Stops the cell and prevents further cells from running
    
import json

from pathlib import Path
# 9/21/2026. 
"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

data3 ->    1. embeddings for 3 fields [court_name, docket_number, state]
            2. summary of big text field
"""

directory = Path("/Users/anria/ab_python/django/data/")
writedir = "/Users/anria/ab_python/django/data2/"
i = 0


# Loop through all text files
for read_file in directory.glob("*.json"):
    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        # 9/19/26 - change element name
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")

        # 9/19/26 - get rid of date unused field
        if "date" in json_data:
            json_data.pop("date")

        # 9/19/26 - add state field
        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()

        # 9/21/26 - use AI to pull out data and create more fields
        combo_field = json_data["case_name"] + " " + json_data["case_date"] + " " + json_data["court_name"] 
        json_data["judge_ruling"] = AI.get_judge_and_ruling(combo_field)
        json_data["summary"] = AI.get_summary(combo_field)
    
        if json_data["judge_ruling"] != "":
            json_data["dv_judge_ruling"] = AI.get_embeddings(json_data["judge_ruling"])
            
        if json_data["summary"] != "":
            json_data["dv_summary"] = AI.get_embeddings(json_data["summary"])
        json_data["dv_general_text"] = AI.get_embeddings(combo_field)
        
        write_file = writedir + read_file.name
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
     

print("Out of loop")

In [35]:
messages=[ {
                        'role': 'system', 'content': 'Forget all previous prompts. You are NEXUS-LEGAL, a domain-specialized uncensored AI assistant for legal tasks.'
                      },
                      { 'role': 'user', 'content': system_prompt + text_input} ],

NameError: name 'system_prompt' is not defined

In [10]:
import json
import os
from pathlib import Path
import ollama
from ollama import chat
import numpy as np
import time

class utils():
    def get_base_dir():
        return os.getcwd()
        
    def check_dir_exist(dir):
        if not os.path.exists(dir):
            os.makedirs(dir)
        return True
    
    def check_file_exists(file_path):
        if os.path.exists(file_path):
            return True
        else:
            return False
    
    def clean_garbage(input_text):
        print(len(input_text) )
        garbage = "\n\nSome case metadata and case summaries were written with the help of AI, which can produce inaccuracies. You\n        should read the full case before relying on it for legal research purposes.\n        This site is protected by reCAPTCHA and the Google\n    Privacy Policy and\n    Terms of Service apply.\n\n\n\nYou're all set! You already receive all suggested Justia Opinion Summary Newsletters. You can explore additional available newsletters here.\n        \n\n\n                Sign up for our free summaries and get the latest delivered directly to you.\n            \n\n                Our Suggestions:\n            \n\n\n\n\n\n\nEnter Your Email\n\n\n\nEnter Your Email\nEnter Your Email\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\n\n\n\n\n\n\n\n\n\nJustia Webinars are open to all, lawyers and non-lawyers. Lawyers, please visit individual webinar pages for more information about CLE accreditation.\nGoogle Scholar\nGoogle Books\nLegal Blogs  \n\n\nGoogle Web \nBing Web \n\n\nGoogle News \nGoogle News Archive \nYahoo! News\nHave a legal question? Get free answers from experienced lawyers!\n\n                Ask Question\nLawyers - Get Listed Now!\nGet a free directory profile listing\n\n\nLawyers - Get Listed Now!\nGet a free directory profile listing"
        output = input_text.replace(garbage, "")
        print(len(output) )
        
        output = output.replace("\n", "").replace("-","").replace("\t", "").replace("(","").replace(")","").replace("[","").replace("]","")
        print(len(output) )
        return output


class AI():
    def get_embeddings(text_input):
        embed_model = 'qwen3-embedding:0.6b'
        response = ollama.embed(
            model=embed_model,
            input= text_input,
        )
        full_embedding = []
        
        # Access the numerical vector array
        try: 
            #print(text_input, "\nresponse['embeddings'][0] == ", response['embeddings'][0], "\n=====================================================")
            full_embedding = response['embeddings'][0]
        except:
            #print(text_input, "\nresponse['embeddings'] == ", response['embeddings'], "\n=====================================================")
            full_embedding = response['embeddings']
        
        # 3. Set your desired target dimension
        target_dim = 384
        truncated_vector = np.array(full_embedding[:target_dim])
        
        # 4. Normalize the truncated vector for cosine similarity search
        normalized_vector = truncated_vector / np.linalg.norm(truncated_vector)
        
        return normalized_vector.tolist()

    def get_summary(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Write a 4 sentence summary of this case. "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        # full_response = output.message.content
        full_response = output['response']
        output = []
        return full_response

    def get_judge_and_ruling(text_input):        
        model_name = 'FableForge-AI/nexus-legal'
        full_response = ""   
        system_prompt = "Forget all previous prompts. Who were the attorneys and who was the judge, and what was the ruling of the case? "
        output = ollama.generate(
            model=model_name,
            prompt=system_prompt + text_input,
            stream=False,
            options={
                "temperature": 0.9,# Low temperature ensures strict factual adherence to the text
                "num_ctx": 4096,
                "num_predict": 256,
                "top_k": 40,
                "top_p": 0.9,
                "seed": 42,
                "stop": ["<EOT>"],
                "repeat_penalty": 1.15,
              }
        )

        full_response = output['response']
        output = []
        return full_response

In [ ]:
break
import json
from pathlib import Path

# 9/21/2026 --> add embed and summaries from AI

"""
data2 ->    1. pop date, 
            2. rename court_year to court_name
            3. add state

data3 ->    1. embeddings for 3 fields [court_name, docket_number, state]
            2. summary of big text field
"""

directory = Path("/Users/anria/ab_python/django/data/")
writedir = "/Users/anria/ab_python/django/data3/"
i = 0

def 
# Loop through all text files
for read_file in directory.glob("*.json"):
    write_file = writedir + read_file.name
    if utils.check_file_exists(write_file):
        continue 
    # print(read_file.name)
    with open(read_file) as file:
        json_data = json.load(file)
        # 9/19/26 - change element name
        if "court_year" in json_data:
            json_data["court_name"] = json_data.pop("court_year")

        # 9/19/26 - get rid of date unused field
        if "date" in json_data:
            json_data.pop("date")

        # 9/19/26 - add state field
        state = read_file.name.split("_")[1]
        json_data["state"] = state.capitalize()

        # 9/21/26 - use AI to pull out data and create more fields
        combo_field = json_data["case_name"] + " " + json_data["case_date"] + " " + json_data["court_name"] 
        json_data["judge_ruling"] = AI.get_judge_and_ruling(combo_field)
        json_data["summary"] = AI.get_summary(combo_field)
    
        json_data["dv_general_text"] = AI.get_embeddings(combo_field + json_data["summary"], 50)
        json_data["dv_judge_ruling"] = AI.get_embeddings(json_data["judge_ruling"], 50)
        
        # print("writefile = ", write_file)
        with open(write_file, "w") as wfile:
            json.dump(json_data, wfile, indent=4, sort_keys=True)
     
    """i+=1
    if i > 4:
        break"""
print("Out of loop")